<a href="https://colab.research.google.com/github/jacielefreitas63-tech/projeto-delivery-logistic/blob/main/1_limpeza_e_eda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧪 Projeto Delivery: Engenharia de Dados & Análise Exploratória (EDA)
> *Etapa 1:* Configuração do Ambiente e Conexão com o Data Lake (Google Drive)

Neste notebook, realizaremos a ingestão, limpeza fina, tratamento de outliers e união das bases transacionais da *Olist, malha rodoviária do **OpenStreetMap* e dados meteorológicos do *INMET*.

In [1]:
# Importação das bibliotecas fundamentais para manipulação e análise estatística
import pandas as pd
import numpy as np

print("🚀 Bibliotecas carregadas com sucesso!")

🚀 Bibliotecas carregadas com sucesso!


In [2]:
# Conexão segura com o armazenamento de dados do Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 📦 1. Ingestão da Base de Pedidos (Olist Orders)
> O objetivo desta etapa é ler o dataset transacional de ordens de serviço, inspecionar os tipos de dados primitivos e mapear o volume inicial de registros antes de aplicar as regras de limpeza.

In [6]:
# Definir o caminho do arquivo no seu Drive (ajustado com _)
caminho_orders = "/content/drive/MyDrive/projeto_delivery_logistica/data/olist_orders_dataset.csv"

# Carregar o arquivo usando o Pandas
df_orders = pd.read_csv(caminho_orders)

# Exibir a volumetria do dataset (Linhas, Colunas)
print(f"Dimensões do Dataset: {df_orders.shape[0]} linhas e {df_orders.shape[1]} colunas.\n")

# Mostrar as primeiras 5 linhas para inspeção visual
df_orders.head()

Dimensões do Dataset: 99441 linhas e 8 colunas.



,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [7]:
# Verificar os tipos de dados de cada coluna e a presença de valores nulos
df_orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4   order_approved_at              99281 non-null  object
 5   order_delivered_carrier_date   97658 non-null  object
 6   order_delivered_customer_date  96476 non-null  object
 7   order_estimated_delivery_date  99441 non-null  object
dtypes: object(8)
memory usage: 6.1+ MB


## 🛠️ 2. Conversão de Tipos (Cast) e Tratamento de Datas
> As colunas temporais de carimbo de data/hora (timestamps) precisam ser convertidas de object para datetime64. Isso nos permitirá calcular intervalos de tempo, como o tempo de entrega real (SLA) e atrasos.

In [8]:
# Lista de colunas que contêm datas no dataset
colunas_datas = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

# Converter todas as colunas da lista para o formato datetime
for coluna in colunas_datas:
    df_orders[coluna] = pd.to_datetime(df_orders[coluna])

# Validar se a conversão funcionou checando os novos tipos
df_orders[colunas_datas].dtypes

,0
order_purchase_timestamp,datetime64[ns]
order_approved_at,datetime64[ns]
order_delivered_carrier_date,datetime64[ns]
order_delivered_customer_date,datetime64[ns]
order_estimated_delivery_date,datetime64[ns]


## 🔍 3. Análise de Valores Ausentes (Missing Values)
> Antes de qualquer modelagem preditiva, mapeamos o volume absoluto e percentual de nulos para decidir entre estratégias de imputação ou remoção de registros.

In [9]:
# Calcular o total de nulos por coluna
total_nulos = df_orders.isnull().sum()

# Calcular a porcentagem de nulos por coluna
porcentagem_nulos = (df_orders.isnull().sum() / len(df_orders)) * 100

# Juntar as informações em um DataFrame de diagnóstico para o relatório
diagnostico_nulos = pd.DataFrame({'Total Nulos': total_nulos, 'Porcentagem (%)': porcentagem_nulos})
diagnostico_nulos.sort_values(by='Total Nulos', ascending=False)

,Total Nulos,Porcentagem (%)
order_delivered_customer_date,2965,2.981668
order_delivered_carrier_date,1783,1.793023
order_approved_at,160,0.160899
order_id,0,0.000000
order_purchase_timestamp,0,0.000000
order_status,0,0.000000
customer_id,0,0.000000
order_estimated_delivery_date,0,0.000000


## 🚚 4. Engenharia de Atributos (Feature Engineering): Métricas de SLA
> Criaremos as variáveis preditivas e de performance logística:
> 1. **tempo_entrega_dias**: Tempo real gasto desde a compra até a chegada ao cliente.
> 2. **atraso_entrega_dias**: Dias de atraso caso a entrega passe do prazo estimado (valores positivos indicam quebra de SLA).

In [10]:
# 1. Calcular o tempo de entrega real em dias
df_orders['tempo_entrega_dias'] = (df_orders['order_delivered_customer_date'] - df_orders['order_purchase_timestamp']).dt.total_seconds() / 86400

# 2. Calcular o atraso em relação à estimativa (Valores > 0 significam atraso)
df_orders['atraso_entrega_dias'] = (df_orders['order_delivered_customer_date'] - df_orders['order_estimated_delivery_date']).dt.total_seconds() / 86400

# Substituir valores negativos por 0 no atraso (pois entregas antes do prazo não têm atraso)
df_orders['atraso_entrega_dias'] = df_orders['atraso_entrega_dias'].clip(lower=0)

# Exibir uma amostragem com as novas colunas calculadas
df_orders[['order_id', 'tempo_entrega_dias', 'atraso_entrega_dias']].head()

,order_id,tempo_entrega_dias,atraso_entrega_dias
0,e481f51cbdc54678b7cc49136f2d6af7,8.436574,0.0
1,53cdb2fc8bc7dce0b6741e2150273451,13.782037,0.0
2,47770eb9100c2d0c44946d9cf07ec65d,9.394213,0.0
3,949d5b44dbf5de918fe9c16f97b45f8a,13.208750,0.0
4,ad21c59c0840e6cb83a9ceb5573f8159,2.873877,0.0


## 🌤️ 5. Ingestão e Tratamento de Dados Meteorológicos (INMET)
> O objetivo desta etapa é carregar o histórico climático, tratar valores nulos de precipitação (chuva) e preparar a base para cruzamento temporal com os momentos de compra e entrega.

In [13]:
# Mapear o caminho correto com o nome do arquivo da região Sudeste
caminho_clima = "/content/drive/MyDrive/projeto_delivery_logistica/data/southeast.csv"

# Ler os dados do INMET (Agora ativado sem o #)
df_clima = pd.read_csv(caminho_clima)

# Exibir a volumetria (Linhas, Colunas)
print(f"Dimensões do Dataset de Clima: {df_clima.shape[0]} linhas e {df_clima.shape[1]} colunas.\n")

# Mostrar as primeiras linhas
df_clima.head()

Dimensões do Dataset de Clima: 15345216 linhas e 27 colunas.



,index,Data,Hora,"PRECIPITAÇÃO TOTAL, HORÁRIO (mm)","PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)",PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB),PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB),RADIACAO GLOBAL (Kj/m²),"TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)",TEMPERATURA DO PONTO DE ORVALHO (°C),...,"VENTO, DIREÇÃO HORARIA (gr) (° (gr))","VENTO, RAJADA MAXIMA (m/s)","VENTO, VELOCIDADE HORARIA (m/s)",region,state,station,station_code,latitude,longitude,height
0,0,2000-05-07,00:00,-9999.0,-9999.0,-9999.0,-9999.0,-9999,-9999.0,-9999.0,...,-9999,-9999.0,-9999.0,SE,RJ,ECOLOGIA AGRICOLA,A601,-22.8,-43.683333,33.0
1,1,2000-05-07,01:00,-9999.0,-9999.0,-9999.0,-9999.0,-9999,-9999.0,-9999.0,...,-9999,-9999.0,-9999.0,SE,RJ,ECOLOGIA AGRICOLA,A601,-22.8,-43.683333,33.0
2,2,2000-05-07,02:00,-9999.0,-9999.0,-9999.0,-9999.0,-9999,-9999.0,-9999.0,...,-9999,-9999.0,-9999.0,SE,RJ,ECOLOGIA AGRICOLA,A601,-22.8,-43.683333,33.0
3,3,2000-05-07,03:00,-9999.0,-9999.0,-9999.0,-9999.0,-9999,-9999.0,-9999.0,...,-9999,-9999.0,-9999.0,SE,RJ,ECOLOGIA AGRICOLA,A601,-22.8,-43.683333,33.0
4,4,2000-05-07,04:00,-9999.0,-9999.0,-9999.0,-9999.0,-9999,-9999.0,-9999.0,...,-9999,-9999.0,-9999.0,SE,RJ,ECOLOGIA AGRICOLA,A601,-22.8,-43.683333,33.0


In [14]:
# 1. Filtrar o dataset para conter apenas dados do estado de SP
df_clima_sp = df_clima[df_clima['state'] == 'SP'].copy()

# 2. Substituir os códigos de erro (-9999.0 e 9999.0) por NaN (nulo oficial do Pandas)
colunas_numericas = df_clima_sp.select_dtypes(include=[np.number]).columns
df_clima_sp[colunas_numericas] = df_clima_sp[colunas_numericas].replace([-9999.0, 9999.0], np.nan)

# 3. Verificar como ficou a limpeza e a nova volumetria
print(f"Volume após filtrar por SP: {df_clima_sp.shape[0]} linhas.")
df_clima_sp.head()

Volume após filtrar por SP: 4288560 linhas.


,index,Data,Hora,"PRECIPITAÇÃO TOTAL, HORÁRIO (mm)","PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)",PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB),PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB),RADIACAO GLOBAL (Kj/m²),"TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)",TEMPERATURA DO PONTO DE ORVALHO (°C),...,"VENTO, DIREÇÃO HORARIA (gr) (° (gr))","VENTO, RAJADA MAXIMA (m/s)","VENTO, VELOCIDADE HORARIA (m/s)",region,state,station,station_code,latitude,longitude,height
5736,0.0,2001-08-30,00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,SE,SP,BAURU,A705,-22.358056,-49.028889,666.0
5737,1.0,2001-08-30,01:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,SE,SP,BAURU,A705,-22.358056,-49.028889,666.0
5738,2.0,2001-08-30,02:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,SE,SP,BAURU,A705,-22.358056,-49.028889,666.0
5739,3.0,2001-08-30,03:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,SE,SP,BAURU,A705,-22.358056,-49.028889,666.0
5740,4.0,2001-08-30,04:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,SE,SP,BAURU,A705,-22.358056,-49.028889,666.0


## 🗺️ 6. Ingestão e Tratamento de Dados de Malha Rodoviária (OpenStreetMap)
> Nesta etapa, mapeamos as coordenadas geográficas (latitude e longitude) dos clientes e vendedores para calcular as distâncias reais das rotas e entender o impacto do tráfego e das rodovias no tempo de entrega.

In [17]:
# Mapear o caminho dos dados de rotas/geolocalização no seu Drive
caminho_rotas = "/content/drive/MyDrive/projeto_delivery_logistica/data/olist_geolocation_dataset.csv"

# Ler os dados de geolocalização que vieram junto com o pacote da Olist
df_geolocation = pd.read_csv(caminho_rotas)

# Exibir a volumetria (Linhas, Colunas)
print(f"Dimensões do Dataset de Geolocalização: {df_geolocation.shape[0]} linhas e {df_geolocation.shape[1]} colunas.\n")

# Mostrar as primeiras linhas para checar a estrutura de lat/lng
df_geolocation.head()

Dimensões do Dataset de Geolocalização: 1000163 linhas e 5 colunas.



,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP


## 🔄 7. Consolidação do Giga Dataset (Merge das Fontes)
> Nesta etapa, realizamos o cruzamento das três fontes de dados (Pedidos, Clima e Geolocalização) utilizando chaves temporais e espaciais para criar uma base unificada focada no estado de São Paulo.

In [20]:
# Listar o nome real de todas as colunas do clima
list(df_clima_sp.columns)

['INDEX',
 'DATA',
 'HORA',
 'PRECIPITAÇÃO TOTAL, HORÁRIO (MM)',
 'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (MB)',
 'PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (MB)',
 'PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (MB)',
 'RADIACAO GLOBAL (KJ/M²)',
 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)',
 'TEMPERATURA DO PONTO DE ORVALHO (°C)',
 'TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)',
 'TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C)',
 'TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (°C)',
 'TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (°C)',
 'UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)',
 'UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)',
 'UMIDADE RELATIVA DO AR, HORARIA (%)',
 'VENTO, DIREÇÃO HORARIA (GR) (° (GR))',
 'VENTO, RAJADA MAXIMA (M/S)',
 'VENTO, VELOCIDADE HORARIA (M/S)',
 'REGION',
 'STATE',
 'STATION',
 'STATION_CODE',
 'LATITUDE',
 'LONGITUDE',
 'HEIGHT']

In [21]:
# 1. Ajustar formato da data na tabela de pedidos
df_orders['order_purchase_timestamp'] = pd.to_datetime(df_orders['order_purchase_timestamp'])
df_orders['data_compra'] = df_orders['order_purchase_timestamp'].dt.date

# 2. Garantir que as colunas do clima estão em maiúsculo
df_clima_sp.columns = [col.upper() for col in df_clima_sp.columns]

# 3. Definir os nomes exatos mapeados nos seus prints
col_chuva = 'PRECIPITAÇÃO TOTAL, HORÁRIO (MM)'
col_temp = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'

# 4. Ajustar tipo da data no clima e fazer o agrupamento diário (média)
df_clima_sp['DATA'] = pd.to_datetime(df_clima_sp['DATA']).dt.date
clima_diario = df_clima_sp.groupby('DATA')[[col_chuva, col_temp]].mean().reset_index()

# 5. O Grande Merge: Unir os Pedidos da Olist com o Clima do INMET
df_consolidado = pd.merge(df_orders, clima_diario, left_on='data_compra', right_on='DATA', how='inner')

# 6. Exibir volumetria e resultado na tela
print(f"Volume do Dataset Consolidado: {df_consolidado.shape[0]} linhas.")
df_consolidado.head()

Volume do Dataset Consolidado: 99441 linhas.


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,tempo_entrega_dias,atraso_entrega_dias,data_compra,DATA,"PRECIPITAÇÃO TOTAL, HORÁRIO (MM)","TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)"
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,8.436574,0.0,2017-10-02,2017-10-02,1.026402,19.678207
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,13.782037,0.0,2018-07-24,2018-07-24,0.005804,19.317283
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,9.394213,0.0,2018-08-08,2018-08-08,0.057674,18.257930
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,13.208750,0.0,2017-11-18,2017-11-18,0.449543,22.989667
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,2.873877,0.0,2018-02-13,2018-02-13,0.471566,22.764302


In [25]:
# Listar o nome de todas as colunas de pedidos disponíveis
list(df_orders.columns)

['order_id',
 'customer_id',
 'order_status',
 'order_purchase_timestamp',
 'order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date',
 'order_estimated_delivery_date',
 'tempo_entrega_dias',
 'atraso_entrega_dias',
 'data_compra']

In [27]:
#1. Carregar a base de clientes (que faz a ponte entre o pedido e o CEP)
caminho_clientes = "/content/drive/MyDrive/projeto_delivery_logistica/data/olist_customers_dataset.csv"
df_customers = pd.read_csv(caminho_clientes)

# 2. Unir a base de pedidos (df_orders) com a de clientes para trazer o CEP
df_orders_com_cep = pd.merge(df_orders, df_customers[['customer_id', 'customer_zip_code_prefix']], on='customer_id', how='inner')

# 3. Preparar a tabela de clima com agrupamento diário
df_clima_sp.columns = [col.upper() for col in df_clima_sp.columns]
col_chuva = 'PRECIPITAÇÃO TOTAL, HORÁRIO (MM)'
col_temp = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'
df_clima_sp['DATA'] = pd.to_datetime(df_clima_sp['DATA']).dt.date
clima_diario = df_clima_sp.groupby('DATA')[[col_chuva, col_temp]].mean().reset_index()

# 4. Primeiro Merge: Pedidos + Clima
df_orders_com_cep['data_compra'] = pd.to_datetime(df_orders_com_cep['order_purchase_timestamp']).dt.date
df_consolidado = pd.merge(df_orders_com_cep, clima_diario, left_on='data_compra', right_on='DATA', how='inner')

# 5. Agrupar a geolocalização por CEP para evitar duplicar linhas
geo_agrupado = df_geolocation_sp.groupby('geolocation_zip_code_prefix')[['geolocation_lat', 'geolocation_lng']].mean().reset_index()

# 6. Segundo Merge: Incluir as Coordenadas no Giga Dataset
df_giga = pd.merge(df_consolidado, geo_agrupado, left_on='customer_zip_code_prefix', right_on='geolocation_zip_code_prefix', how='inner')

# 7. Exibir volumetria final do Giga Dataset
print(f"Volume do Giga Dataset Final: {df_giga.shape[0]} linhas e {df_giga.shape[1]} colunas.")
df_giga.head()

Volume do Giga Dataset Final: 41731 linhas e 18 colunas.


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,tempo_entrega_dias,atraso_entrega_dias,data_compra,customer_zip_code_prefix,DATA,"PRECIPITAÇÃO TOTAL, HORÁRIO (MM)","TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)",geolocation_zip_code_prefix,geolocation_lat,geolocation_lng
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,8.436574,0.0,2017-10-02,3149,2017-10-02,1.026402,19.678207,3149,-23.576983,-46.587161
1,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,2.873877,0.0,2018-02-13,9195,2018-02-13,0.471566,22.764302,9195,-23.676370,-46.514627
2,e69bfb5eb88e0ed6a785585b27e16dbf,31ad1d1b63eb9962463f764d4e6e0c9d,delivered,2017-07-29 11:55:02,2017-07-29 12:05:32,2017-08-10 19:45:24,2017-08-16 17:14:30,2017-08-23,18.221852,0.0,2017-07-29,18075,2017-07-29,0.001159,18.417149,18075,-23.474030,-47.467397
3,34513ce0c4fab462a55830c0989c7edb,7711cf624183d843aafe81855097bc37,delivered,2017-07-13 19:58:11,2017-07-13 20:10:08,2017-07-14 18:43:29,2017-07-19 14:04:48,2017-08-08,5.754595,0.0,2017-07-13,4278,2017-07-13,0.002576,18.450234,4278,-23.601856,-46.608910
4,5ff96c15d0b717ac6ad1f3d77225a350,19402a48fe860416adf93348aba37740,delivered,2018-07-25 17:44:10,2018-07-25 17:55:14,2018-07-26 13:16:00,2018-07-30 15:52:25,2018-08-08,4.922396,0.0,2018-07-25,4812,2018-07-25,0.012094,19.129989,4812,-23.713190,-46.687407
